In [70]:
import pandas as pd

In [71]:
df = pd.read_csv("vehicles_dataset.csv")
df.head()

,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state
0,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         426880 non-null  int64  
 1   year          425675 non-null  float64
 2   manufacturer  409234 non-null  str    
 3   model         421603 non-null  str    
 4   condition     252776 non-null  str    
 5   cylinders     249202 non-null  str    
 6   fuel          423867 non-null  str    
 7   odometer      422480 non-null  float64
 8   title_status  418638 non-null  str    
 9   transmission  424324 non-null  str    
 10  drive         296313 non-null  str    
 11  size          120519 non-null  str    
 12  type          334022 non-null  str    
 13  paint_color   296677 non-null  str    
 14  state         426880 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage: 48.9 MB


In [73]:
df.describe()

,price,year,odometer
count,4.268800e+05,425675.000000,4.224800e+05
mean,7.519903e+04,2011.235191,9.804333e+04
std,1.218228e+07,9.452120,2.138815e+05
min,0.000000e+00,1900.000000,0.000000e+00
25%,5.900000e+03,2008.000000,3.770400e+04
50%,1.395000e+04,2013.000000,8.554800e+04
75%,2.648575e+04,2017.000000,1.335425e+05
max,3.736929e+09,2022.000000,1.000000e+07


In [74]:
df = df[
    (df["price"] >= 500) &
    (df["price"] <= 100000)
]

In [75]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [76]:
df.info()

<class 'pandas.DataFrame'>
Index: 384131 entries, 0 to 426879
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         384131 non-null  int64  
 1   year          382959 non-null  float64
 2   manufacturer  368695 non-null  str    
 3   model         379637 non-null  str    
 4   condition     238824 non-null  str    
 5   cylinders     227216 non-null  str    
 6   fuel          381538 non-null  str    
 7   odometer      382016 non-null  float64
 8   title_status  377134 non-null  str    
 9   transmission  382319 non-null  str    
 10  drive         267115 non-null  str    
 11  size          108132 non-null  str    
 12  type          301079 non-null  str    
 13  paint_color   272758 non-null  str    
 14  state         384131 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage: 46.9 MB


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor



X = df.drop("price", axis=1).copy()
y = df["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Target encoding

global_mean = y_train.mean()

for col in ["manufacturer", "model"]:

    # media prezzo per categoria (solo TRAIN)
    target_mean = y_train.groupby(
        X_train[col].fillna("missing")
    ).mean()

    # TRAIN
    X_train[col] = (
        X_train[col]
        .fillna("missing")
        .map(target_mean)
        .fillna(global_mean)
    )

    # TEST
    X_test[col] = (
        X_test[col]
        .fillna("missing")
        .map(target_mean)
        .fillna(global_mean)
    )


# one hot encoding 
onehot_cols = [
    "condition",
    "cylinders",
    "fuel",
    "title_status",
    "transmission",
    "drive",
    "size",
    "type",
    "paint_color",
    "state"
]

X_train = pd.get_dummies(
    X_train,
    columns=onehot_cols,
    dummy_na=True
)

X_test = pd.get_dummies(
    X_test,
    columns=onehot_cols,
    dummy_na=True
)


X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)


# gestione nulli
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# bool -> int
for col in X_train.columns:
    if X_train[col].dtype == bool:
        X_train[col] = X_train[col].astype(int)
        X_test[col] = X_test[col].astype(int)

# definizione del modello

model = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

# metriche di correttezza 

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- RISULTATI VALUTAZIONE ---")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAE  : {mae:,.2f}")
print(f"R²   : {r2:.4f} ({r2*100:.2f}%)")


--- RISULTATI VALUTAZIONE ---
MSE  : 22,025,102.00
RMSE : 4,693.09
MAE  : 2,587.19
R²   : 0.8932 (89.32%)


In [91]:
X_train

,year,manufacturer,model,odometer,condition_excellent,condition_fair,condition_good,condition_like new,condition_new,condition_salvage,...,state_tn,state_tx,state_ut,state_va,state_vt,state_wa,state_wi,state_wv,state_wy,state_nan
113806,2012.0,13240.593964,8877.058885,139453.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
263923,2008.0,21389.125587,7416.359459,140800.0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
304267,2014.0,21265.247108,18430.108911,148000.0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59806,1988.0,17447.931298,19843.587533,128400.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
66638,2009.0,13240.593964,4950.000000,109.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286095,2006.0,12184.403201,6281.381818,70230.0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
407140,2013.0,25133.624000,15500.000000,90602.0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
147100,2018.0,16618.661033,30315.000000,20595.0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
162882,2004.0,20235.994424,24102.875927,260000.0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
